# PYTORCH(YOLO) 2 ONNX 

In [1]:
from ultralytics import YOLO 

In [ ]:
model = YOLO("yolov8l.pt")

results = model.predict( source="city.jpg", project="./", name="yolo" , save=True)

for result in results :
    result.save(filename="result.jpg")

### convert YOLO model to ONNX :

In [11]:
model.export(format="onnx")

Ultralytics YOLOv8.0.220 🚀 Python-3.10.1 torch-2.1.1+cpu CPU (11th Gen Intel Core(TM) i5-11300H 3.10GHz)

PyTorch: starting from 'yolov8l.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (83.7 MB)

ONNX: starting export with onnx 1.15.0 opset 17...
ONNX: export success ✅ 3.7s, saved as 'yolov8l.onnx' (166.8 MB)

Export complete (7.2s)
Results saved to C:\Users\Lenovo\Desktop\onnxcode
Predict:         yolo predict task=detect model=yolov8l.onnx imgsz=640  
Validate:        yolo val task=detect model=yolov8l.onnx imgsz=640 data=coco.yaml  
Visualize:       https://netron.app


'yolov8l.onnx'

In [4]:
# use this onnx model :
onnx_model = YOLO("models/yolov8l.onnx" , task='detect')
result = onnx_model("city.jpg")

Loading models\yolov8l.onnx for ONNX Runtime inference...

image 1/1 c:\Users\Lenovo\Desktop\onnxcode\city.jpg: 640x640 12 persons, 3 cars, 2 buss, 2 traffic lights, 1 tie, 702.3ms
Speed: 38.8ms preprocess, 702.3ms inference, 47.9ms postprocess per image at shape (1, 3, 640, 640)


### for installing ultralytics , it needs to install lots of libraries like torch , nvidia-cudnn , ...

### but in onnx , we only need to install onnxruntime 

# ---------------------------------------------------------

# scikit-learn

In [5]:
import numpy as np 
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

iris_dataset = load_iris()
X , Y = iris_dataset.data , iris_dataset.target
X = X.astype(np.float32)

X_trian , X_test , Y_trian , Y_test = train_test_split(X , Y)

model = RandomForestClassifier()
model.fit(X_trian , Y_trian)

RandomForestClassifier()

### convert scikit-learn model to ONNX model : 

In [ ]:
!pip install skl2onnx


'''
where to find these libraries like skl2onnx ?

https://github.com/onnx 

for sklearn     :  https://github.com/onnx/sklearn-onnx
for tensorflow  :  https://github.com/onnx/tensorflow-onnx

'''



### when a sklearn model is converting to ONNX model : 
1_ we have to send << one of our input-data>> as a sample , to model  , in order to model find out the input shape , or input datatype , and etc ... . 

In [6]:
X[0]

array([        5.1,         3.5,         1.4,         0.2], dtype=float32)

In [7]:
import skl2onnx

# skl2onnx.to_onnx( model , one-sample-input-data )
onnx_model = skl2onnx.to_onnx( model=model , X=X[0] )

with open("models/sklearn2onnx_model.onnx" , "wb") as file :
    file.write(onnx_model.SerializeToString())


# -------------------------------------------------------------------


# pytorch to ONNX :
https://pytorch.org/tutorials/beginner/onnx/export_simple_model_to_onnx_tutorial.html

!pip install onnxscript

In [8]:
import torch
# import torch.nn as NN
# import torch.nn.functional as F

class MyModel(torch.nn.Module):
    def __init__(self):
        super(MyModel  , self).__init__()
        self.conv1 = torch.nn.Conv2d(in_channels=1 , out_channels=6 ,kernel_size=5)
        self.conv2 = torch.nn.Conv2d(in_channels=6 , out_channels=16 , kernel_size=5)
        self.fc1   = torch.nn.Linear(in_features= 16*5*5 , out_features= 120)
        self.fc2   = torch.nn.Linear(in_features=120 , out_features=84)
        self.fc3   = torch.nn.Linear(in_features=84 , out_features=10)

    def forward(self , x) :
        x =  torch.nn.functional.max_pool2d(input=torch.nn.functional.relu(self.conv1(x)) , kernel_size=(2,2))
        x =  torch.nn.functional.max_pool2d(input=torch.nn.functional.relu(input=self.conv2(x)) , kernel_size=2)
        x =  torch.flatten(input=x , start_dim=1)
        x =  torch.nn.functional.relu(input=self.fc1(x))
        x =  torch.nn.functional.relu(input=self.fc2(x))
        x =  self.fc3(x)
        return x
    


# convert TORCH to ONNX :
torch_model = MyModel()
sample_input = torch.randn(1,1,32,32) # batchsize , channel , imagesize
onnx_model = torch.onnx.export( torch_model , sample_input , "models/torch2onnxmodel.onnx") # model + one_sample_input
# onnx_model = torch.onnx.dynamo_export(torch_model, sample_input)
# onnx_model.save("torch2onnxmodel.onnx")

In [10]:
# checking saved model 
import onnx
onnx_model = onnx.load("models/torch2onnxmodel.onnx")
onnx.checker.check_model(onnx_model)


# --------------------------------------------------------------------------

# TENSORFLOW 2 ONNX :


https://github.com/onnx/tensorflow-onnx

In [ ]:
!pip install tf2onnx

In [ ]:
import numpy as np 
from keras.applications.resnet50 import ResNet50
import tf2onnx
import tensorflow as tf 
import keras
import onnx
import tf_keras

model = keras.applications.resnet50.ResNet50(weights="imagenet" , include_top=True )
onnx_model  , x = tf2onnx.convert.from_keras(model , [tf.TensorSpec(shape=model.inputs[0].shape , dtype=model.inputs[0].dtype  , name=model.inputs[0].name )] ) # should pass model & input-sample

onnx.save(onnx_model , "models/tensorflow2onnx.onnx")